# HarrisLabPlotting — P-Value Plotting Tutorial (Notebook)

This notebook is the runnable companion to **`PVALUE_PLOTTING_TUTORIAL.md`**.
It walks through:

1. Generating a random p-value matrix from the 28-ROI tutorial network
   (`connectivity_28.edge` + `rois_28.node`) and saving it under
   `test_files/tutorial_files/node_edge_28/` so it ships with the repo.
2. Plotting the p-value matrix on the brain mesh using
   `create_brain_connectivity_plot` with `matrix_type='pvalue'`.
3. The same plot but **signed**, using a sign matrix so positive
   effects render in red and negative effects in blue.
4. The same data with `create_brain_connectivity_plot_with_modularity`
   so you can verify the module-deselection fix and per-module
   p-value visualisation.
5. The CLI equivalent of every Python call (one shell-cell per
   plot) so you can copy-paste them into a terminal.
6. The `transform_pvalue_matrix` helper called directly so you can
   inspect the `-log10(p)` weight matrix on its own.

**Run this notebook from anywhere** — every path is resolved relative
to the repo root using `pathlib`.

> See the markdown tutorial `PVALUE_PLOTTING_TUTORIAL.md` for the full
> flag reference and a longer discussion of why we use `-log10(p)`.

## 1. Setup — locate paths and import the package

We resolve every input path **relative to the repo root** so this
notebook works whether you launched Jupyter from `tutorial/`, the
repo root, or somewhere else.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Walk up from this notebook to the repo root (the directory that
# contains 'test_files' and 'tutorial').
here = Path.cwd()
repo_root = here
for p in [here, *here.parents]:
    if (p / 'test_files').is_dir() and (p / 'tutorial').is_dir():
        repo_root = p
        break
print('Repo root:', repo_root)

TUTORIAL_FILES = repo_root / 'test_files' / 'tutorial_files'
NODE_EDGE_28 = TUTORIAL_FILES / 'node_edge_28'
OUTPUT_DIR = repo_root / 'tutorial' / 'output_pvalue'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir:', OUTPUT_DIR)

from HarrisLabPlotting import (
    load_mesh_file,
    create_brain_connectivity_plot,
    create_brain_connectivity_plot_with_modularity,
    transform_pvalue_matrix,
    load_edge_color_matrix,
    filter_matrix_by_sign,
)

## 2. Generate the random p-value matrix from `connectivity_28.edge`

We map each absolute connection strength to a small p-value (stronger
edges → smaller p) and add a noise term so significance lines up with
where the original 28-ROI network has connections. The result is
saved both as `.csv` and `.npy` so users can pick whichever format
they prefer. We also save a **sign matrix** that records the sign of
the underlying connection — that's what enables signed colouring
(positive effects in red, negative in blue) in section 5.

These files are committed to the repo (`pvalues_28.csv`,
`pvalues_28.npy`, `pvalues_28_signs.csv`, `pvalues_28_signs.npy`)
so most readers can skip this cell — it's idempotent and safe to
re-run if you want a different random seed.

In [ ]:
edge_path = NODE_EDGE_28 / 'connectivity_28.edge'
node_path = NODE_EDGE_28 / 'rois_28.node'

mat = np.loadtxt(edge_path, delimiter='\t')
n = mat.shape[0]
print('connectivity_28.edge shape:', mat.shape)

rng = np.random.default_rng(seed=42)
abs_w = np.abs(mat)
norm = abs_w / max(abs_w.max(), 1.0)

# Stronger edges -> smaller p
base = (1.0 - 0.95 * norm) ** 5
noise = rng.uniform(0.0005, 1.0, size=mat.shape)
p = base * noise
p = (p + p.T) / 2          # symmetric
np.fill_diagonal(p, 1.0)
p[mat == 0] = 1.0          # no edge -> p = 1 (will be filtered out)
p = np.clip(p, 1e-6, 1.0)

sign = np.sign(mat).astype(int)

np.save(NODE_EDGE_28 / 'pvalues_28.npy', p)
pd.DataFrame(p).to_csv(NODE_EDGE_28 / 'pvalues_28.csv', index=False, header=False)
np.save(NODE_EDGE_28 / 'pvalues_28_signs.npy', sign)
pd.DataFrame(sign).to_csv(NODE_EDGE_28 / 'pvalues_28_signs.csv', index=False, header=False)

nz = p[p < 1.0]
print('Saved pvalues_28.csv / .npy and pvalues_28_signs.csv / .npy')
print(f'  edges with p < 1   : {nz.size}')
print(f'  edges with p < 0.05: {int(((p < 0.05) & (p > 0)).sum())}')
print(f'  edges with p < 0.01: {int(((p < 0.01) & (p > 0)).sum())}')
print(f'  median p (non-1)   : {np.median(nz):.4f}')
print(f'  min p              : {nz.min():.2g}')

## 3. Inspect the `-log10(p)` transform on its own

`transform_pvalue_matrix` is the helper the plot functions call under
the hood. It's exported at the top level of the package so you can
use it directly to build a weight matrix for your own analyses.

In [ ]:
p_demo = np.array([
    [1.0,   0.001, 0.04, 0.20],
    [0.001, 1.0,   0.50, 0.07],
    [0.04,  0.50,  1.0,  0.001],
    [0.20,  0.07,  0.001, 1.0],
])

weights, p_clean = transform_pvalue_matrix(p_demo, pvalue_threshold=0.05)
print('Input p-values:')
print(p_demo)
print('\n-log10(p) with p > 0.05 dropped:')
print(np.round(weights, 3))
print('\np_clean (input p with dropped cells set to 0):')
print(p_clean)

## 4. Plot the p-value matrix on the brain mesh

We need a coordinates file for the 28 ROIs. The standard CLI tutorial
(see `tutorial/CLI_TUTORIAL.md`) generates one at
`output/atlas_28_mapped/atlas_28_mapped_comma.csv`. To make this
notebook **self-contained**, the next cell will fall back to the
pre-generated `atlas_28_test_comma.csv` that already lives in
`test_files/tutorial_files/output/`.

In [ ]:
candidates = [
    TUTORIAL_FILES / 'output' / 'atlas_28_mapped' / 'atlas_28_mapped_comma.csv',
    TUTORIAL_FILES / 'output' / 'atlas_28_test_comma.csv',
]
coords_path = next((c for c in candidates if c.exists()), None)
if coords_path is None:
    raise FileNotFoundError(
        'No 28-ROI coordinates file found. Run `hlplot coords map-subset` '
        'or follow steps 1-4 of CLI_TUTORIAL.md first.'
    )
print('Using coords:', coords_path)

vertices, faces = load_mesh_file(str(TUTORIAL_FILES / 'brain_mesh.gii'))
coords_df = pd.read_csv(coords_path)
print('coords cols:', list(coords_df.columns))
print('n_rois     :', len(coords_df))

In [ ]:
# 4a. Basic p-value plot — single color, default threshold of 0.05
fig, stats = create_brain_connectivity_plot(
    vertices=vertices,
    faces=faces,
    roi_coords_df=coords_df,
    connectivity_matrix=str(NODE_EDGE_28 / 'pvalues_28.csv'),
    matrix_type='pvalue',
    pvalue_threshold=0.05,
    edge_width=(1.0, 8.0),
    plot_title='28-ROI p-value network (p \u2264 0.05)',
    save_path=str(OUTPUT_DIR / 'pval_basic.html'),
    camera_view='superior',
    
)
print('edges drawn:', stats['total_edges'])

In [ ]:
fig.show()

In [ ]:
# 4b. Stricter threshold: only p \u2264 0.01
fig, stats = create_brain_connectivity_plot(
    vertices=vertices,
    faces=faces,
    roi_coords_df=coords_df,
    connectivity_matrix=str(NODE_EDGE_28 / 'pvalues_28.csv'),
    matrix_type='pvalue',
    pvalue_threshold=0.01,
    edge_width=(1.0, 8.0),
    plot_title='28-ROI p-value network (p \u2264 0.01)',
    save_path=str(OUTPUT_DIR / 'pval_strict.html'),
    camera_view='superior',
)
print('edges drawn at p<=0.01:', stats['total_edges'])

In [ ]:
fig.show()

## 5. Signed p-values — pos/neg coloring with `--sign-matrix`

Pass the saved sign matrix as `sign_matrix` and `hlplot` will draw
edges whose underlying effect was positive in `pos_edge_color` and
edges whose effect was negative in `neg_edge_color`.

In [ ]:
fig, stats = create_brain_connectivity_plot(
    vertices=vertices,
    faces=faces,
    roi_coords_df=coords_df,
    connectivity_matrix=str(NODE_EDGE_28 / 'pvalues_28.csv'),
    matrix_type='pvalue',
    pvalue_threshold=0.05,
    sign_matrix=str(NODE_EDGE_28 / 'pvalues_28_signs.csv'),
    pos_edge_color='#d62728',
    neg_edge_color='#1f77b4',
    edge_width=(1.0, 8.0),
    plot_title='Signed p-values (p \u2264 0.05)',
    save_path=str(OUTPUT_DIR / 'pval_signed.html'),
    camera_view='oblique',
)
print('total edges        :', stats['total_edges'])
print('positive (red)     :', stats['positive_edges'])
print('negative (blue)    :', stats['negative_edges'])

In [ ]:
fig.show()

## 6. Modularity p-value plot

We use a tiny synthetic module assignment for the 28 ROIs
(`(np.arange(28) % 4) + 1`) just so the modular plot has groups to
visualise. Real users would pass their own community-detection output.

Click on a module entry in the plot legend — both the module's nodes
**and** edges should disappear together (this is the bug fix from this
release).

In [ ]:
modules = (np.arange(len(coords_df)) % 4) + 1
modules_path = NODE_EDGE_28 / 'modules_28.csv'
pd.DataFrame({'module': modules}).to_csv(modules_path, index=False)
print('Saved synthetic modules to', modules_path)

fig, stats = create_brain_connectivity_plot_with_modularity(
    vertices=vertices,
    faces=faces,
    roi_coords_df=coords_df,
    connectivity_matrix=str(NODE_EDGE_28 / 'pvalues_28.csv'),
    module_assignments=modules,
    matrix_type='pvalue',
    pvalue_threshold=0.05,
    sign_matrix=str(NODE_EDGE_28 / 'pvalues_28_signs.csv'),
    edge_color_mode='sign',
    pos_edge_color='#d62728',
    neg_edge_color='#1f77b4',
    edge_width=(1.0, 8.0),
    plot_title='28-ROI p-value modularity (signed)',
    save_path=str(OUTPUT_DIR / 'pval_modular_signed.html'),
    camera_view='oblique',
)
print('n_modules :', stats['n_modules'])
print('edges     :', stats['total_edges'])

In [ ]:
fig.show()

In [ ]:
# Same data but edges colored by source-node module instead of by sign.
fig, stats = create_brain_connectivity_plot_with_modularity(
    vertices=vertices,
    faces=faces,
    roi_coords_df=coords_df,
    connectivity_matrix=str(NODE_EDGE_28 / 'pvalues_28.csv'),
    module_assignments=modules,
    matrix_type='pvalue',
    pvalue_threshold=0.05,
    edge_color_mode='module',
    edge_width=(1.0, 8.0),
    plot_title='28-ROI p-value modularity (by module)',
    save_path=str(OUTPUT_DIR / 'pval_modular_module.html'),
    camera_view='oblique',
)

In [ ]:
fig.show()

## 7. Per-edge colors via `edge_color_matrix`

Sometimes you want to color edges by something that isn't sign and
isn't module — for example a custom community label, or a manually
highlighted sub-network. `edge_color_matrix` is a same-shape matrix
where each cell is **its own color** (or an integer label that maps
to a palette). It overrides both `pos_edge_color`/`neg_edge_color` and
the module coloring.

Below we build an integer-label matrix that classifies each
significant p-value edge into one of three groups by significance
tier and let `hlplot` pick the colors.

In [ ]:
p = np.load(NODE_EDGE_28 / 'pvalues_28.npy')
labels = np.zeros_like(p, dtype=int)
labels[(p > 0) & (p <= 0.05)] = 1   # significant
labels[(p > 0) & (p <= 0.01)] = 2   # very significant
labels[(p > 0) & (p <= 0.001)] = 3  # extremely significant
labels_path = NODE_EDGE_28 / 'pvalues_28_tier_labels.csv'
pd.DataFrame(labels).to_csv(labels_path, index=False, header=False)
print('saved', labels_path)

_, label_to_color = load_edge_color_matrix(str(labels_path), n_expected_nodes=28)
print('label -> color:', label_to_color)

fig, stats = create_brain_connectivity_plot(
    vertices=vertices,
    faces=faces,
    roi_coords_df=coords_df,
    connectivity_matrix=str(NODE_EDGE_28 / 'pvalues_28.csv'),
    matrix_type='pvalue',
    pvalue_threshold=0.05,
    edge_color_matrix=str(labels_path),
    edge_width=(1.0, 8.0),
    plot_title='28-ROI p-values colored by significance tier',
    save_path=str(OUTPUT_DIR / 'pval_tier_colored.html'),
    camera_view='superior',
)
print('edges drawn:', stats['total_edges'])

## 8. CLI equivalents

The cells below are shell commands. They reproduce the Python plots
above using the `hlplot` CLI. They are wrapped in `!` so the notebook
can shell out — but you can also paste any of them straight into a
terminal.

All CLI commands assume your working directory is the repo root.

In [ ]:
# 8a. Quickstart -- equivalent of section 4a
os.chdir(repo_root)
!hlplot plot \
  --mesh test_files/tutorial_files/brain_mesh.gii \
  --coords "{coords_path}" \
  --matrix test_files/tutorial_files/node_edge_28/pvalues_28.csv \
  --matrix-type pvalue \
  --pvalue-threshold 0.05 \
  --edge-width-min 1 --edge-width-max 8 \
  --output tutorial/output_pvalue/cli_pval_basic.html \
  --title "28-ROI p-value network (CLI)" \
  --camera superior

In [ ]:
# 8b. Signed p-values -- equivalent of section 5
!hlplot plot \
  --mesh test_files/tutorial_files/brain_mesh.gii \
  --coords "{coords_path}" \
  --matrix test_files/tutorial_files/node_edge_28/pvalues_28.csv \
  --matrix-type pvalue \
  --sign-matrix test_files/tutorial_files/node_edge_28/pvalues_28_signs.csv \
  --pvalue-threshold 0.05 \
  --pos-edge-color "#d62728" \
  --neg-edge-color "#1f77b4" \
  --edge-width-min 1 --edge-width-max 8 \
  --output tutorial/output_pvalue/cli_pval_signed.html \
  --title "Signed p-values (CLI)" \
  --camera oblique

In [ ]:
# 8c. Modularity p-value plot -- equivalent of section 6
!hlplot modular \
  --mesh test_files/tutorial_files/brain_mesh.gii \
  --coords "{coords_path}" \
  --matrix test_files/tutorial_files/node_edge_28/pvalues_28.csv \
  --modules test_files/tutorial_files/node_edge_28/modules_28.csv \
  --matrix-type pvalue \
  --sign-matrix test_files/tutorial_files/node_edge_28/pvalues_28_signs.csv \
  --pvalue-threshold 0.05 \
  --edge-color-mode sign \
  --pos-edge-color "#d62728" \
  --neg-edge-color "#1f77b4" \
  --edge-width-min 1 --edge-width-max 8 \
  --output tutorial/output_pvalue/cli_pval_modular_signed.html \
  --title "Signed p-value modularity (CLI)"

In [ ]:
# 8d. Per-edge color matrix from significance tiers -- equivalent of section 7
!hlplot plot \
  --mesh test_files/tutorial_files/brain_mesh.gii \
  --coords "{coords_path}" \
  --matrix test_files/tutorial_files/node_edge_28/pvalues_28.csv \
  --matrix-type pvalue \
  --pvalue-threshold 0.05 \
  --edge-color-matrix test_files/tutorial_files/node_edge_28/pvalues_28_tier_labels.csv \
  --edge-width-min 1 --edge-width-max 8 \
  --output tutorial/output_pvalue/cli_pval_tier_colored.html \
  --title "Significance tiers (CLI)" \
  --camera superior

## 9. Edge width multiplier — `edge_width_scale`

Sometimes you've already tuned `edge_width=(min, max)` to taste and
you just want every edge to be **uniformly thicker** (or thinner) for
a particular figure — for example, to make a poster legible from a
distance, or to dial down width for a PDF print. The
`edge_width_scale` parameter is a single multiplier applied AFTER all
other scaling. It works for both `edge_width=(min, max)` and
`edge_width=fixed_value`, and on both the basic plot and the
modularity plot. The CLI exposes the same option as
`--edge-width-scale`.

| Setting | Effect |
| --- | --- |
| `edge_width=(1, 5), edge_width_scale=1` | widths in [1, 5] (default) |
| `edge_width=(1, 5), edge_width_scale=5` | widths in [5, 25] |
| `edge_width=2.0, edge_width_scale=5`    | every edge has width 10 |

In [ ]:
# Same data as section 6, but every edge is drawn 5x thicker via
# edge_width_scale. The signed coloring and per-module legend grouping
# all still work normally.
fig, stats = create_brain_connectivity_plot_with_modularity(
    vertices=vertices,
    faces=faces,
    roi_coords_df=coords_df,
    connectivity_matrix=str(NODE_EDGE_28 / 'pvalues_28.csv'),
    module_assignments=modules,
    matrix_type='pvalue',
    pvalue_threshold=0.05,
    sign_matrix=str(NODE_EDGE_28 / 'pvalues_28_signs.csv'),
    edge_color_mode='sign',
    pos_edge_color='#d62728',
    neg_edge_color='#1f77b4',
    edge_width=(1.0, 5.0),
    edge_width_scale=5.0,        # <-- 5x multiplier
    plot_title='Signed p-values, edges 5x thicker',
    save_path=str(OUTPUT_DIR / 'pval_modular_signed_5x.html'),
    camera_view='oblique',
)
fig.show()

In [ ]:
# And the equivalent CLI invocation
!hlplot modular \
  --mesh test_files/tutorial_files/brain_mesh.gii \
  --coords "{coords_path}" \
  --matrix test_files/tutorial_files/node_edge_28/pvalues_28.csv \
  --modules test_files/tutorial_files/node_edge_28/modules_28.csv \
  --matrix-type pvalue \
  --sign-matrix test_files/tutorial_files/node_edge_28/pvalues_28_signs.csv \
  --pvalue-threshold 0.05 \
  --edge-color-mode sign \
  --edge-width-min 1 --edge-width-max 5 \
  --edge-width-scale 5 \
  --output tutorial/output_pvalue/cli_pval_modular_signed_5x.html \
  --title "Signed p-value modularity 5x (CLI)"

## 10. Brain mesh lighting / shininess

The brain mesh is a `plotly.graph_objects.Mesh3d`, which natively
supports physically-inspired lighting controls (`ambient`, `diffuse`,
`specular`, `roughness`, `fresnel`). `hlplot` exposes these as a
high-level **preset** (`mesh_style`) plus optional **raw knobs** so
you can override individual values on top of the preset.

Default behaviour is **unchanged** -- pass nothing and the mesh looks
exactly like in previous releases. The presets are:

| `mesh_style` | look |
| --- | --- |
| `'default'` (or `None`) | legacy (no extra lighting) |
| `'flat'`   | paper cutout, no shading |
| `'matte'`  | chalk, no specular highlight |
| `'smooth'` | balanced, slight specular and rim |
| `'glossy'` | shiny plastic with sharp highlight |
| `'mirror'` | near-chrome (max specular, near-zero roughness) |

Edges and **node markers do not get lit** -- plotly does not run
lighting on `Scatter3d` markers/lines, so the lighting flags only
affect the brain mesh.


In [ ]:
# 10a. The 'glossy' preset on a signed p-value plot
fig, _ = create_brain_connectivity_plot(
    vertices=vertices,
    faces=faces,
    roi_coords_df=coords_df,
    connectivity_matrix=str(NODE_EDGE_28 / 'pvalues_28.csv'),
    matrix_type='pvalue',
    pvalue_threshold=0.05,
    sign_matrix=str(NODE_EDGE_28 / 'pvalues_28_signs.csv'),
    pos_edge_color='#d62728',
    neg_edge_color='#1f77b4',
    edge_width=(1.0, 8.0),
    mesh_opacity=0.25,            # bump up so the gloss is visible
    mesh_style='mirror',          # <-- one-flag preset
    plot_title='Signed p-values, glossy mesh',
    save_path=str(OUTPUT_DIR / 'pval_mesh_glossy.html'),
    camera_view='oblique',
)
fig.show()


In [ ]:
# 10b. Glossy preset + raw specular override (cranked higher)
fig, _ = create_brain_connectivity_plot(
    vertices=vertices,
    faces=faces,
    roi_coords_df=coords_df,
    connectivity_matrix=str(NODE_EDGE_28 / 'pvalues_28.csv'),
    matrix_type='pvalue',
    pvalue_threshold=0.05,
    sign_matrix=str(NODE_EDGE_28 / 'pvalues_28_signs.csv'),
    edge_width=(1.0, 8.0),
    mesh_opacity=0.30,
    mesh_style='glossy',
    mesh_specular=2.0,            # <-- override the preset's specular only
    mesh_roughness=0.1,           # <-- and make it sharper
    mesh_light_position=(2.0, 2.0, 1.5),
    plot_title='Glossy preset + specular=2.0, roughness=0.1',
    save_path=str(OUTPUT_DIR / 'pval_mesh_glossy_override.html'),
    camera_view='oblique',
)
fig.show()


In [ ]:
# 10c. CLI equivalents
!hlplot plot \
  --mesh test_files/tutorial_files/brain_mesh.gii \
  --coords "{coords_path}" \
  --matrix test_files/tutorial_files/node_edge_28/pvalues_28.csv \
  --matrix-type pvalue \
  --sign-matrix test_files/tutorial_files/node_edge_28/pvalues_28_signs.csv \
  --pvalue-threshold 0.05 \
  --edge-width-min 1 --edge-width-max 8 \
  --mesh-opacity 0.25 \
  --mesh-style glossy \
  --output tutorial/output_pvalue/cli_pval_mesh_glossy.html \
  --title "Signed p-values, glossy mesh (CLI)"

!hlplot plot \
  --mesh test_files/tutorial_files/brain_mesh.gii \
  --coords "{coords_path}" \
  --matrix test_files/tutorial_files/node_edge_28/pvalues_28.csv \
  --matrix-type pvalue \
  --sign-matrix test_files/tutorial_files/node_edge_28/pvalues_28_signs.csv \
  --pvalue-threshold 0.05 \
  --edge-width-min 1 --edge-width-max 8 \
  --mesh-opacity 0.30 \
  --mesh-style glossy --mesh-specular 2.0 --mesh-roughness 0.1 \
  --mesh-light-position "2,2,1.5" \
  --output tutorial/output_pvalue/cli_pval_mesh_glossy_override.html \
  --title "Glossy + override (CLI)"


## 11. Custom camera + live xyz readout

Two related features that work together:

1. **Custom camera input.** Pass an explicit eye / center / up + a name
   to open the plot at *exactly* that view. The custom view is appended
   to the camera dropdown so you can flip back to it any time, and any
   `--export-image` PNG/SVG/PDF inherits the same camera.

2. **Live camera readout.** When `show_camera_readout=True`, a small
   fixed-position overlay is injected into the saved HTML that updates
   in real time as you rotate the brain. It shows the current
   `eye` / `center` / `up`, plus a copy-pastable block of CLI flags
   that reproduce the current view. The overlay is **HTML-only** -- it
   never appears in static image exports because those are rendered
   server-side by kaleido. Pass `show_camera_readout=False` (or simply
   omit it) if you do **not** want the overlay in your HTML.

The intended workflow:

1. Render an interactive plot once with `show_camera_readout=True`.
2. Drag the brain around in the browser until you find the angle you
   want.
3. Copy the `--custom-camera-eye / --center / --up / --name` block
   from the overlay.
4. Paste it into your final command (now with
   `show_camera_readout=False` and an `--export-image`) to bake that
   view into a clean PNG/SVG/PDF.


In [ ]:
# 11a. Render with the live readout enabled. Open the HTML in a browser
# and drag the brain around -- the overlay in the top-right corner
# updates in real time and prints the CLI flags you need to reproduce
# the current view.
fig, _ = create_brain_connectivity_plot(
    vertices=vertices,
    faces=faces,
    roi_coords_df=coords_df,
    connectivity_matrix=str(NODE_EDGE_28 / 'pvalues_28.csv'),
    matrix_type='pvalue',
    pvalue_threshold=0.05,
    sign_matrix=str(NODE_EDGE_28 / 'pvalues_28_signs.csv'),
    edge_width=(1.0, 8.0),
    plot_title='Drag me to find your angle',
    save_path=str(OUTPUT_DIR / 'pval_camera_readout.html'),
    show_camera_readout=True,    # <-- live xyz overlay in the saved HTML
)
print('Open', OUTPUT_DIR / 'pval_camera_readout.html',
      'in a browser and rotate.')


In [ ]:
fig.show()

In [ ]:
# 11b. Render at an exact custom view (and give it a name).
# These are made-up numbers -- in real use you would have copied them
# from the overlay in the previous cell.
fig, _ = create_brain_connectivity_plot(
    vertices=vertices,
    faces=faces,
    roi_coords_df=coords_df,
    connectivity_matrix=str(NODE_EDGE_28 / 'pvalues_28.csv'),
    matrix_type='pvalue',
    pvalue_threshold=0.05,
    sign_matrix=str(NODE_EDGE_28 / 'pvalues_28_signs.csv'),
    edge_width=(1.0, 8.0),
    custom_camera=dict(
        eye=dict(x=1.8, y=0.6, z=1.2),
        center=dict(x=0, y=0, z=0),
        up=dict(x=0, y=0, z=1),
    ),
    custom_camera_name='Right-Anterior 3/4',
    show_camera_readout=False,    # OFF for the final clean output
    plot_title='Saved view for the paper',
    save_path=str(OUTPUT_DIR / 'pval_camera_custom.html'),
    export_image=str(OUTPUT_DIR / 'pval_camera_custom.png'),
    image_dpi=200,
    export_show_title=False,
    export_show_legend=False,
)
fig.show()


In [ ]:
# 11c. CLI equivalents
# Step 1: render with the live readout to interactively pick an angle.
!hlplot plot \
  --mesh test_files/tutorial_files/brain_mesh.gii \
  --coords "{coords_path}" \
  --matrix test_files/tutorial_files/node_edge_28/pvalues_28.csv \
  --matrix-type pvalue \
  --sign-matrix test_files/tutorial_files/node_edge_28/pvalues_28_signs.csv \
  --pvalue-threshold 0.05 \
  --edge-width-min 1 --edge-width-max 8 \
  --show-camera-readout \
  --output tutorial/output_pvalue/cli_camera_readout.html \
  --title "Drag me"

# Step 2: rerun with the explicit custom camera (no readout, with image export).
!hlplot plot \
  --mesh test_files/tutorial_files/brain_mesh.gii \
  --coords "{coords_path}" \
  --matrix test_files/tutorial_files/node_edge_28/pvalues_28.csv \
  --matrix-type pvalue \
  --sign-matrix test_files/tutorial_files/node_edge_28/pvalues_28_signs.csv \
  --pvalue-threshold 0.05 \
  --edge-width-min 1 --edge-width-max 8 \
  --custom-camera-eye "1.8,0.6,1.2" \
  --custom-camera-center "0,0,0" \
  --custom-camera-up "0,0,1" \
  --custom-camera-name "Right-Anterior 3/4" \
  --no-camera-readout \
  --output tutorial/output_pvalue/cli_camera_custom.html \
  --export-image tutorial/output_pvalue/cli_camera_custom.png \
  --image-dpi 200 \
  --export-no-title --export-no-legend \
  --title "Saved view for the paper (CLI)"


### Same thing on the modular plot

Both features work identically on `hlplot modular` /
`create_brain_connectivity_plot_with_modularity`. The custom view is
appended to the dropdown alongside the built-in presets, and clicking
a module entry in the legend still toggles the module's nodes and
edges in lockstep.


In [ ]:
fig, _ = create_brain_connectivity_plot_with_modularity(
    vertices=vertices,
    faces=faces,
    roi_coords_df=coords_df,
    connectivity_matrix=str(NODE_EDGE_28 / 'pvalues_28.csv'),
    module_assignments=modules,
    matrix_type='pvalue',
    pvalue_threshold=0.05,
    sign_matrix=str(NODE_EDGE_28 / 'pvalues_28_signs.csv'),
    edge_color_mode='sign',
    edge_width=(1.0, 8.0),
    mesh_opacity=0.25,
    mesh_style='glossy',
    custom_camera=dict(
        eye=dict(x=1.8, y=0.6, z=1.2),
        center=dict(x=0, y=0, z=0),
        up=dict(x=0, y=0, z=1),
    ),
    custom_camera_name='Right-Anterior 3/4',
    show_camera_readout=True,
    plot_title='Modular pvalue + glossy + saved camera + readout',
    save_path=str(OUTPUT_DIR / 'pval_modular_camera_readout.html'),
)
fig.show()


## 12. Where to go from here

- **Markdown reference:** `tutorial/PVALUE_PLOTTING_TUTORIAL.md` lists
  every relevant flag with explanations.
- **General CLI tutorial:** `tutorial/CLI_TUTORIAL.md` covers the rest
  of `hlplot` (coordinates, weighted matrices, exports, batch mode).
- **Helper API:** `transform_pvalue_matrix`, `load_edge_color_matrix`,
  and `filter_matrix_by_sign` are all importable from
  `HarrisLabPlotting`.
- **CLI thresholding:** the new `hlplot utils threshold --keep-sign`
  flag lets you split a weighted matrix into a positive-only or
  negative-only sub-matrix before plotting (useful when you want to
  visualise excitatory vs inhibitory subnets independently).
- **Edge width multiplier:** the new `edge_width_scale` /
  `--edge-width-scale` knob is the easiest way to make every edge
  proportionally thicker without rebalancing your `min`/`max`.